# N128 · 模运算、快速幂与逆元

**目标：** 在合法代数条件下化简大数计算。

**先修：** N126, N124, N063。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 模同余；快速幂；互素逆元；Fermat前提；CRT扩展；复合模数。

## 从问题到算法

模运算中加减乘可直接化简，除法不行：只有与模数互素的元素才有逆元。快速幂按指数二进制分解，每次平方底数、处理最低指数位。超长十进制指数不必转换为一个巨大整数，读入新位d后把原结果提升十次方再乘a^d。

## 最小实现

**本章接口：** `mod_pow(a, n, mod)`、`mod_inverse(a, mod)`、`super_pow(a, digits)`

In [1]:
def mod_pow(a,n,mod):
    if n<0 or mod<=0: raise ValueError('nonnegative exponent and positive modulus required')
    result=1%mod; a%=mod
    while n:
        if n&1: result=result*a%mod
        a=a*a%mod; n>>=1
    return result

def mod_inverse(a,mod):
    if mod<2: raise ValueError('inverse modulus must be at least 2')
    old_r,r=a%mod,mod; old_x,x=1,0
    while r:
        q=old_r//r; old_r,r=r,old_r-q*r; old_x,x=x,old_x-q*x
    if old_r!=1: raise ValueError('inverse does not exist: gcd != 1')
    return old_x%mod

def super_pow(a,digits):
    answer=1
    for digit in digits:
        if not isinstance(digit,int) or not 0<=digit<=9: raise ValueError('decimal exponent digits required')
        answer=mod_pow(answer,10,1337)*mod_pow(a,digit,1337)%1337
    return answer

## 正确性、前提与复杂度

快速幂保持result·a^n与原目标同余，处理奇偶后不变量不变。扩展GCD给出ax+my=g，只有g=1时x才是逆元。指数逐位更新利用a^(10e+d)=(a^e)^10a^d，对任意正模数成立，不依赖模数为素数。

**代价：** 快速幂O(log n)次模乘；逆元O(log mod)级别除法；D位十进制指数O(D)次固定小指数模幂。均需计入大整数乘除位复杂度；不盲用费马小定理处理复合模数。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

a,n,mod=3,13,17; result=1; rows=[]
while n:
    rows.append((n,a,result,n&1))
    if n&1: result=result*a%mod
    a=a*a%mod; n>>=1
show_table(['剩余指数','当前底数','累计结果','是否相乘'],rows)
print('最终结果',result)

剩余指数,当前底数,累计结果,是否相乘
13,3,1,1
6,9,3,0
3,13,3,1
1,16,5,1


最终结果 12


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
import math
assert mod_inverse(3,11)==4 and mod_pow(4,0,1)==0
for a in range(-12,13):
    for n in range(20):
        for mod in range(1,15): assert mod_pow(a,n,mod)==pow(a,n,mod)
    for mod in range(2,20):
        if math.gcd(a,mod)==1: assert a*mod_inverse(a,mod)%mod==1
        else:
            try: mod_inverse(a,mod)
            except ValueError: pass
            else: raise AssertionError('noninvertible value accepted')
    for n in range(80): assert super_pow(a,list(map(int,str(n))))==pow(a,n,1337)
assert super_pow(2,[])==1
print('N128: 所有本章断言通过')

N128: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 比较扩展欧几里得求逆和素数模快速幂求逆。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 构造不可逆元素。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 50. Pow(x, n)（comparison）
- 372. Super Pow（canonical）
- 1922. Count Good Numbers（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。